# 7. Spatial joins: attaching census geography

We now have:

- a library location dataset;
- a census polygon dataset.

Unlike the ward join from Week 3, there is **no shared census ID in the library data**.

We therefore join based on location.

## Attribute join versus spatial join

### Attribute join

Two tables share an identifier:

| Library | Ward code |
|---|---:|
| Branch A | 03 |
| Branch B | 11 |

joins to:

| Ward code | Ward name |
|---|---|
| 03 | Etobicoke–Lakeshore |
| 11 | University–Rosedale |

Matching happens because the values in `Ward code` are equal.

### Spatial join

A location table has no census ID:

| Library | Geometry |
|---|---|
| Branch A | POINT(...) |
| Branch B | POINT(...) |

A polygon table does:

| ADAUID | Geometry |
|---|---|
| 35200001 | POLYGON(...) |
| 35200002 | POLYGON(...) |

We ask:

> Which polygon contains each point?

GeoPandas then attaches the polygon's attributes to that point.

## Visual intuition

```text
Census area A              Census area B
┌─────────────────┐        ┌─────────────────┐
│      ● Library  │        │                 │
│                 │        │   ● Library     │
│   ● Library     │        │                 │
└─────────────────┘        └─────────────────┘

          ↓ spatial join ↓

Library 1 → Area A
Library 2 → Area A
Library 3 → Area B
```

In [ ]:
from pathlib import Path

import geopandas as gpd

In [ ]:
libraries_file = Path("../data/processed/toronto_libraries_clean.gpkg")
practice_file = Path("../data/practice/toronto_library_points_practice.geojson")

if libraries_file.exists():
    libraries = gpd.read_file(libraries_file)
else:
    libraries = gpd.read_file(practice_file)

census = gpd.read_file(
    "../data/raw/toronto_census_areas.geojson"
)

## Step 1: use the same CRS

A spatial join compares geometries, so both layers must describe coordinates in the same CRS.

In [ ]:
print("Libraries CRS:", libraries.crs)
print("Census CRS:", census.crs)

libraries = libraries.to_crs(census.crs)

## Step 2: create one point per feature

Some OSM features may be polygons.

For this introductory workflow, `representative_point()` gives each feature one point that lies within its geometry.

In [ ]:
library_points = libraries.copy()

library_points["geometry"] = (
    library_points.geometry.representative_point()
)

## Step 3: perform the spatial join

`predicate="within"` means:

> keep the census-area attributes where the library point is within that polygon.

In [ ]:
libraries_with_census = gpd.sjoin(
    library_points,
    census[
        [
            "ADAUID",
            "pop_2021_count",
            "hh_no_veh_pct",
            "geometry"
        ]
    ],
    how="left",
    predicate="within"
)

display(libraries_with_census.head())

## Check the join

A successful command is not automatically a correct analysis.

Compare row counts and inspect missing census IDs.

In [ ]:
print("Rows before:", len(library_points))
print("Rows after:", len(libraries_with_census))
print(
    "Missing ADAUID:",
    libraries_with_census["ADAUID"].isna().sum()
)